# S6 - Feature engineering

Raw BTS columns → model features. Horizon: **we predict before departure**, so no "at-departure / post-flight" column may enter as a feature (leakage).

This notebook is a **prototype** - once the features stabilize, the logic moves to `src/flight_delay/transform/features.py` as functions (Phase B just wraps them).

In [1]:
from pathlib import Path
import pandas as pd

pd.set_option("display.max_columns", None)

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "pyproject.toml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA = PROJECT_ROOT / "data" / "processed"

df = pd.read_parquet(DATA)
flights = df.loc[(df["Cancelled"] == False) & (df["Diverted"] == False), :].copy().reset_index(drop=True)
flights["y"] = flights["ArrDelay"].ge(15).astype(int)
flights.shape

(6965247, 33)

## Step 1 - column whitelist (leakage boundary)

With leakage it is safer to list what is **allowed** than what is forbidden: a new column drops out by default instead of sneaking in.
Keep `FlightDate` aside - for the time split (S8) and historical features, not as a raw feature. Skip `Tail_Number` for now (high cardinality; may be useful for aircraft-level history later).

In [2]:
# Columns known BEFORE departure - the only ones allowed as features.
SAFE_RAW = [
    "Month", "DayOfWeek",
    "Reporting_Airline",
    "Origin", "Dest", "OriginState", "DestState",
    "CRSDepTime", "CRSArrTime", "CRSElapsedTime",
    "Distance", "DistanceGroup",
]

# At-departure / post-flight columns - LEAKAGE, never features.
LEAKAGE = [
    "DepDelay", "DepDelayMinutes", "DepDel15",
    "TaxiOut", "TaxiIn",
    "ActualElapsedTime", "AirTime",
    "ArrDelay", "ArrDelayMinutes", "ArrDel15",
    "CarrierDelay", "WeatherDelay", "NASDelay",
    "SecurityDelay", "LateAircraftDelay",
    "Cancelled", "Diverted", "CancellationCode",
]

# Sanity-check 1: nothing is both safe and leakage.
assert not (set(SAFE_RAW) & set(LEAKAGE))

# Sanity-check 2: what is left unassigned? (expected: FlightDate, Tail_Number, y)
set(flights.columns) - set(SAFE_RAW) - set(LEAKAGE)

{'FlightDate', 'Tail_Number', 'y'}

## Step 2 - "free" features (straight from raw columns, no aggregation)

Only `DepHour` is **genuinely needed**: `CRSDepTime` is HHMM as a number (1345, 900, 2400) - as an int it is not linear in time (1345→1400 is a jump of 55, not 15), so we decode it to the hour and fix `2400→0`.

`IsWeekend` / `Season` are **partly redundant for XGBoost** (the tree splits `DayOfWeek` / `Month` itself) - but they help the linear baseline in S7, which cannot find thresholds on its own. We build them so both models have what they need.

> `Distance` and `DistanceGroup` are redundant (one is the binned other) - we pick one in S7/S8.

In [3]:
feat = flights[SAFE_RAW].copy()

# Scheduled hour: CRSDepTime = HHMM; 2400 (midnight) -> 0.
feat["DepHour"] = (flights["CRSDepTime"] // 100).replace(24, 0)

# Weekend (DayOfWeek: 1 = Monday ... 7 = Sunday).
feat["IsWeekend"] = flights["DayOfWeek"].isin([6, 7]).astype(int)

# Season from month.
feat["Season"] = flights["Month"].map({
    12: "winter", 1: "winter", 2: "winter",
    3: "spring", 4: "spring", 5: "spring",
    6: "summer", 7: "summer", 8: "summer",
    9: "fall", 10: "fall", 11: "fall",
})

feat["y"] = flights["y"]
feat.head()

,Month,DayOfWeek,Reporting_Airline,Origin,Dest,OriginState,DestState,CRSDepTime,CRSArrTime,CRSElapsedTime,Distance,DistanceGroup,DepHour,IsWeekend,Season,y
0,1,1,9E,LGA,OMA,NY,NE,856,1135,219.0,1148.0,5,8,0,winter,0
1,1,2,9E,LGA,OMA,NY,NE,856,1135,219.0,1148.0,5,8,0,winter,0
2,1,3,9E,LGA,OMA,NY,NE,856,1135,219.0,1148.0,5,8,0,winter,0
3,1,4,9E,LGA,OMA,NY,NE,856,1135,219.0,1148.0,5,8,0,winter,1
4,1,5,9E,LGA,OMA,NY,NE,856,1135,219.0,1148.0,5,8,0,winter,0


## Step 3 - encoding: native categorical

Decision: for high cardinality (`Origin` / `Dest` / `Reporting_Airline`) use **native categorical** in XGBoost (`enable_categorical=True`): no leakage, no column blow-up, optimal subset splits. We don't hard-encode anything - we only set the right dtype and let the model handle it in S8. (Comparison of one-hot / ordinal / native / target encoding: see the chat discussion.)

In [4]:
CATEGORICAL = ["Origin", "Dest", "OriginState", "DestState", "Reporting_Airline", "Season"]
feat[CATEGORICAL] = feat[CATEGORICAL].astype("category")

feat.dtypes

Month                    int8
DayOfWeek                int8
Reporting_Airline    category
Origin               category
Dest                 category
OriginState          category
DestState            category
CRSDepTime              int16
CRSArrTime              int16
CRSElapsedTime        float32
Distance              float32
DistanceGroup            int8
DepHour                 int16
IsWeekend               int64
Season               category
y                       int64
dtype: object